We receive the key cache before any nonlinear functions have been applied, and want to find the original tokens used to generate it. We first note that the key cache was generated using this sequence of steps:

$$
\begin{aligned}
&\text{Token IDs} \;\; x_t \;\;\;\;\;\; \to \;\;\; E[x_t] \in \mathbb{R}^D \\[6pt]
&X = [E[x_1], \dots, E[x_T]] \\[6pt]
&X_n = \mathrm{LayerNorm}(X) \\[6pt]
&K_{\text{flat}} = X_n W_k^\top + b_k \\[6pt]
&K_{\text{cache}} = \mathrm{RoPE}(K_{\text{flat}}) \\[12pt]
\end{aligned}
$$

We can undo the RoPE and use the pseudoinverse of the key weight matrix to work out the pre-normalised embeddings, which we can then layernorm and use a nearest neighbours on to find the original tokens and get the flag.

$$
\begin{aligned}
&K_{\text{flat}} = \mathrm{RoPE}^{-1}(K_{\text{cache}}) \\[6pt]
&X_n \approx (K_{\text{flat}} - b_k) (W_k^\top)^+ \\[6pt]
&\hat{x}_t = \arg\max_j \; \cos\!\left( X_{n,t}, \mathrm{LayerNorm}(E[j]) \right)
\end{aligned}
$$

In [ ]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

ART = Path("dist")
kv  = torch.load(ART/"kv_cache.pt", map_location="cpu")

CKPT, REV = kv["model"], kv["revision"]
DEV = "cpu"

tok   = AutoTokenizer.from_pretrained(CKPT, revision=REV, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(CKPT, revision=REV).to(DEV).eval()

In [ ]:
attn = model.model.layers[0].self_attn
ln0  = model.model.layers[0].input_layernorm
E    = model.get_input_embeddings().weight.detach().cpu()

K_rot = kv["K_rot"].unsqueeze(0)
H, T, Dh = K_rot.shape[1], K_rot.shape[2], K_rot.shape[3]

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

with torch.no_grad():
    position_ids = torch.arange(T, device=K_rot.device).unsqueeze(0)
    cos, sin = attn.rotary_emb(K_rot, position_ids)
    rot_dim = cos.size(-1)

    K_rotblk = K_rot[..., :rot_dim]
    K_tail   = K_rot[...,  rot_dim:]

    K_unrot_first = (K_rotblk * cos) - (rotate_half(K_rotblk) * sin)
    K_unrot = torch.cat([K_unrot_first, K_tail], dim=-1)

In [ ]:
with torch.no_grad():
    K_flat = K_unrot.transpose(1, 2).reshape(T, H * Dh)
    Wk = attn.k_proj.weight.detach().cpu()
    bk = attn.k_proj.bias.detach().cpu() if attn.k_proj.bias is not None else None
    if bk is not None:
        K_flat = K_flat - bk
    Xn = K_flat @ torch.linalg.pinv(Wk.T)

    E_ln = ln0(E)

    Xn_n = Xn / Xn.norm(dim=1, keepdim=True).clamp_min(1e-9)
    E_n  = E_ln / E_ln.norm(dim=1, keepdim=True).clamp_min(1e-9)
    ids  = (Xn_n @ E_n.T).argmax(dim=1).tolist()
    text = tok.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

print(text)

## Sanity Check

In [ ]:
from transformers import DynamicCache

In [ ]:
text = "hello kv cache!"
ids  = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEV)

with torch.no_grad():
    cache = DynamicCache(config=model.config)
    out   = model(input_ids=ids, use_cache=True, past_key_values=cache, return_dict=True)

k0, v0 = out.past_key_values[0]
print("cache:", k0.shape, v0.shape)

In [ ]:
T = ids.shape[1]
K_rot = k0.squeeze(0).contiguous()

In [ ]:
attn = model.model.layers[0].self_attn
H, Dh = attn.num_heads, attn.head_dim
T = ids.shape[1]

with torch.no_grad():
    X = model.get_input_embeddings()(ids)
    Xn = model.model.layers[0].input_layernorm(X)
    K  = attn.k_proj(Xn)
    K  = K.view(1, T, attn.num_key_value_heads, Dh).transpose(1, 2)

    position_ids = torch.arange(T, device=K.device).unsqueeze(0)

    cos, sin = attn.rotary_emb(K, position_ids)
    rot_dim = cos.size(-1)

    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    K_rotblk = K[..., :rot_dim]
    K_tail   = K[...,  rot_dim:]
    K_rot_first = (K_rotblk * cos) + (rotate_half(K_rotblk) * sin)
    K_manual = torch.cat([K_rot_first, K_tail], dim=-1)

In [ ]:
k_real = k0.float().contiguous()
k_reco = K_manual.float().contiguous()
print("match:", torch.allclose(k_real, k_reco, atol=1e-6, rtol=1e-6),
      "max abs diff:", (k_real - k_reco).abs().max().item())